In [ ]:
import pandas as pd
import pathlib as path
raw = path.Path('/Users/apple/Documents/GitHub/SQL/norway_ds_sql/data/raw/Pensionable_income-09855.csv')
df = pd.read_csv(raw, sep=';', encoding='utf-8-sig', skiprows=1, nrows=3)
print(df.shape)
print(df.columns[:5].tolist())
print(df.iloc[:, 0].tolist())

In [ ]:
import pandas as pd
from pathlib import Path
raw = Path('/Users/apple/Documents/GitHub/SQL/norway_ds_sql/data/raw/Pensionable_income-09855.csv')
df = pd.read_csv(raw, sep=';', encoding='utf-8-sig', skiprows=1, na_values=['.', '..', ':', ':;', ''])
df.columns=[c.strip('"') for c in df.columns]
long=df.melt(
    id_vars=['sex'],
    var_name='raw_col', 
    value_name='value')
#long['year']=long['raw_col'].str.extract(r'^(\d{4})').isna().sum()
long['year']=long['raw_col'].str.extract(r'^(\d{4})').astype(int)
long['category']=long['raw_col'].str.replace(r'^\d{4}\s+', '', regex=True) 
print('category_unique:') 
for c in long['category'].unique():
    print(" ", repr(c))
wide=long.pivot_table(
    index=['sex', 'year'],
    columns='category',
    values='value',
    aggfunc='first').reset_index()
wide.columns.name=None
wide=wide.rename(columns=
                 {'Taxable persons with pensionable income': 'taxable_persons',
                 'Average pensionable income (NOK)': 'avg_income_nok',
                 'Median pensionable income (NOK)': 'median_income_nok'})   
wide.year=wide.year.astype(int)
wide.taxable_persons=pd.to_numeric(wide.taxable_persons, errors='coerce').astype('Int64')
wide.avg_income_nok=pd.to_numeric(wide.avg_income_nok, errors='coerce')
wide.median_income_nok=pd.to_numeric(wide.median_income_nok, errors='coerce') 
print(wide.dtypes)
print(wide.isna().sum())
wide=wide.sort_values(['sex', 'year']).reset_index(drop=True)
print(out)
wide.head(6)
print(wide.tail(6))

category_unique:
  'Taxable persons with pensionable income'
  'Average pensionable income (NOK)'
  'Median pensionable income (NOK)'
sex                  object
year                  int64
avg_income_nok        int64
median_income_nok     int64
taxable_persons       Int64
dtype: object
sex                  0
year                 0
avg_income_nok       0
median_income_nok    0
taxable_persons      0
dtype: int64
users/apple/Documents/GitHub/SQL/norway_ds_sql/data/processed/pensionable_income.csv
       sex  year  avg_income_nok  median_income_nok  taxable_persons
168  Males  2019          508600             469100          1964751
169  Males  2020          525300             485700          1946360
170  Males  2021          560200             524700          1834161
171  Males  2022          587400             548300          1872743
172  Males  2023          614500             576000          1893999
173  Males  2024          649700             609400          1902646


In [ ]:
counts=wide.groupby('year')['sex'].count()
print(counts.unique())
counts[counts !=3].sum()
assert (counts ==3).all(), f'not all years have 3 sexes: {counts[counts !=3]}'
assert not wide.duplicated(['sex', 'year']).any(), 'repeated sex.year'
assert (wide['avg_income_nok'].dropna()>0).all() 
assert (wide['median_income_nok'].dropna()>0).all() 
# row=wide[wide['year']==2024].set_index('sex')['taxable_persons']
frow=wide[wide['sex']=='Females'].set_index('year')['taxable_persons']
mrow=wide[wide['sex']=='Males'].set_index('year')['taxable_persons']
brow=wide[wide['sex']=='Both sexes'].set_index('year')['taxable_persons']
diff=brow-(frow+mrow)
print(diff.abs().max())
print(diff[diff.abs() > 0])
dif=abs(frow.sum() + mrow.sum() -brow.sum()) 
diffprosent=dif*100/brow.sum()
print(diffprosent)
print(brow.sum())

[3]
655
year
2024    655
Name: taxable_persons, dtype: Int64
0.0004253312872928687
153997606


In [156]:
out=Path('/Users/apple/Documents/GitHub/SQL/norway_ds_sql/data/processed/pensionable_income.csv')
out.parent.mkdir(parents=True, exist_ok=True)
wide.to_csv(out, index=False, encoding='utf-8-sig')